# 🚀 AGAR-RL V7 : Pipeline SOTA Deep RL (Directional Pursuit, Tactical Split Gating & Log-Scale Threat Awareness)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/train_colab.ipynb)

**Entraînement de haute performance à pleine puissance (GPU L4 / A100) avec auto-sauvegarde Google Drive.**

### 🎯 Nouveautés Majeures de la V7 SOTA (AgarCL & GoBigger Standards) :
1. **Traque Directionnelle Pure (Directional Pursuit Alignment)** : Récompense d'alignement angulaire $\cos(\theta_{\text{action}} - \theta_{\text{proie}}) \times \text{proximité}$. L'agent apprend à rabattre et traquer calmement les proies plus petites sans recourir immédiatement au split.
2. **Perception Log-Ratio des Menaces (Scale-Invariant Observation)** : Vecteur d'observation avec $\tanh(\ln(m_{\text{autre}} / m_{\text{soi}}))$. Différencie immédiatement les proies ($< -0.1$), les rivaux inoffensifs ($+0.1$ à $+0.5$, plus lents et incapables de split-kill) et les prédateurs mortels en split ($> +0.65$), éliminant les panics splits inutiles.
3. **Gating Physique Anti-Suicide & Anti-Popcorn** : Plafond strict à 4 sous-cellules (`max_subcells: 4`) et masse minimale de division à 55 (`min_split_mass: 55.0`). Impossible de s'éparpiller en 11 morceaux sans défense !
4. **Bonus de Frappe Conditionné aux Attaques Mortelles** : Le bonus de split n'est décerné que si la demi-cellule résultante a la masse requise pour avaler la cible ($\frac{m}{2} > 1.15 \times m_{\text{proie}}$). Aucune incitation aux divisions kamikazes.
5. **Zéro Minefield de Pénalités Négatives** : Espace d'optimisation pur et libre d'exploration pour converger vers l'optimum global sans être piégé dans un minimum local timide.
6. **Reprise Transparente depuis V6 (14.5M steps)** : Détection et reprise automatique du dernier checkpoint de `/content/drive/MyDrive/agario_rl_backup_v6` pour continuer l'ascension directement dans `agario_rl_backup_v7` !

## 0. Montage Google Drive & Détection GPU L4
Tous les checkpoints, les replays HD et les modèles ONNX seront automatiquement sauvegardés sur votre Drive dans le dossier `agario_rl_backup_v7`.

In [ ]:
# 1. Montage sécurisé de Google Drive
import os, sys, time, torch

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup_v7'
PREV_BACKUP_V6 = '/content/drive/MyDrive/agario_rl_backup_v6'
PREV_BACKUP_V5 = '/content/drive/MyDrive/agario_rl_backup_v5'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 2. Vérification du matériel accéléré (GPU L4 / A100 recommandé)
print('=' * 65)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'🚀 Accélération GPU Détectée : {gpu_name} ({vram:.1f} Go VRAM)')
    print('⚡ Configuration optimale : 16 environnements parallèles + Numba JIT + Batch 512')
else:
    print('⚠️ Aucun GPU détecté. Activez un GPU dans : Exécution > Modifier le type d\'exécution')
print(f'📁 Dossier Google Drive V7 synchronisé : {DRIVE_BACKUP_DIR}')
print(f'📁 Dossier V6 précédent disponible : {PREV_BACKUP_V6} (Existe: {os.path.exists(PREV_BACKUP_V6)})')
print('=' * 65)

## 1. Synchronisation du Code GitHub & Installation des Dépendances

In [ ]:
import os

# 1. Récupération propre des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Synchronisation avec GitHub main...')
    !git fetch origin main
    !git reset --hard origin/main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et synchronisation avec GitHub main...')
    %cd agario
    !git fetch origin main
    !git reset --hard origin/main
else:
    print('🌐 Clonage propre du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances Farama Gymnasium
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt tensorboard
!apt-get install -qq -y ffmpeg
print('✅ Environnement et dépendances installés avec succès.')

## 2. Validation Pré-Vol : Suite Complète de 29 Tests Unitaires
Vérification complète de la physique du moteur, du remerge magnétique, des récompenses de traque et de l'espace d'observation log-ratio.

In [ ]:
# Exécute tous les tests du moteur physique, du remerge et des récompenses Farama
!python -m pytest -v

## 3. Monitoring TensorBoard (Optionnel)

In [ ]:
import os
os.makedirs('logs/tensorboard', exist_ok=True)
try:
    %load_ext tensorboard
    %tensorboard --logdir logs/tensorboard
except Exception as e:
    print(f'Note TensorBoard : {e}')

## 4. Entraînement Haute Performance V7 (Hunter-Predator Architecture)
- **Reprise Optimale depuis V6 (14.5M steps)** : Détection automatique du plus récent checkpoint dans `agario_rl_backup_v6` pour continuer l'entraînement avec les nouvelles dynamiques de traque et de gating des splits !
- **Sauvegarde Continue V7** : Checkpoints automatiques tous les 250 000 pas dans `agario_rl_backup_v7`.

In [ ]:
# 🚀 Configuration de Reprise & Lancement V7
import os, glob, re

V7_DIR = '/content/drive/MyDrive/agario_rl_backup_v7'
V6_DIR = '/content/drive/MyDrive/agario_rl_backup_v6'
V5_DIR = '/content/drive/MyDrive/agario_rl_backup_v5'
os.makedirs(V7_DIR, exist_ok=True)

def extract_step(path):
    fname = os.path.basename(path)
    if 'final' in fname:
        return 999_999_999
    m = re.search(r'step_(\d+)', fname)
    return int(m.group(1)) if m else 0

# Recherche hiérarchique du meilleur checkpoint : V7 d'abord, puis V6, puis V5
def find_best_checkpoint(dir_path):
    if not os.path.exists(dir_path):
        return None
    zips = glob.glob(os.path.join(dir_path, '*.zip'))
    valid = [z for z in zips if os.path.getsize(z) > 1000 and not os.path.basename(z).startswith('._') and 'bc_pretrained' not in z]
    valid.sort(key=extract_step, reverse=True)
    # Prioritize latest or highest step
    latest = os.path.join(dir_path, 'ppo_latest.zip')
    if valid:
        return valid[0]
    if os.path.exists(latest):
        return latest
    return None

chosen_checkpoint = find_best_checkpoint(V7_DIR) or find_best_checkpoint(V6_DIR) or find_best_checkpoint(V5_DIR)

if chosen_checkpoint:
    resume_flag = f'--resume "{chosen_checkpoint}"'
    step_num = extract_step(chosen_checkpoint)
    print('=' * 75)
    print(f'🎯 Checkpoint source détecté pour reprise : {chosen_checkpoint}')
    print(f'📊 Palier détecté : {step_num:,} steps' if step_num < 999_999_999 else '📊 Palier : FINAL')
    print('=' * 75)
else:
    resume_flag = '--resume auto'
    print('⚠️ Aucun checkpoint préalable trouvé, démarrage d\'un nouvel entraînement.')

!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 20000000 \
    --pool-interval 250000 \
    --backup-dir {V7_DIR} \
    {resume_flag} \
    --device auto

## 5. Inspection Diagnostique de la Politique & Réflexes Tactiques
Sonde le réseau de neurones sur des scénarios synthétiques contrôlés : réponse à la nourriture, esquive des prédateurs mortels vs calme face aux rivaux inoffensifs, et propension à attaquer les proies en zone de frappe.

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v7/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v6/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip')

candidates = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._') and 'bc_pretrained' not in c]
candidates.sort(key=extract_step, reverse=True)
target_inspect = candidates[0] if candidates else 'checkpoints/ppo/ppo_latest.zip'

print('=' * 75)
print(f'🔬 Inspection Diagnostique du Modèle : {target_inspect}')
print('=' * 75)

!python src/analysis/inspect_policy.py --model "{target_inspect}"

## 6. Enregistrement Automatique du Match Replay HD & Visualisation Directe
Génère une vidéo HD de 80 secondes (2400 steps @ 30 FPS) avec affichage tête haute (HUD), vecteurs de décision et radar.

In [ ]:
import os, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v7/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v6/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip')

valid_cands = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._') and 'bc_pretrained' not in c]
valid_cands.sort(key=extract_step, reverse=True)
target_model = valid_cands[0] if valid_cands else 'checkpoints/ppo/ppo_latest.zip'
step_count = extract_step(target_model)

print('=' * 75)
print(f'🎬 Modèle sélectionné pour le Replay HD : {target_model}')
print(f'📊 Palier : {step_count:,} steps')
print('=' * 75)

os.makedirs('recordings', exist_ok=True)
!python src/inference/record_match.py \
    --model "{target_model}" \
    --output recordings/eval_match_v7.mp4 \
    --steps 2400

if os.path.exists('recordings/eval_match_v7.mp4') and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v7'):
    !cp recordings/eval_match_v7.mp4 /content/drive/MyDrive/agario_rl_backup_v7/eval_match_v7.mp4
    print('📁 Replay HD copié sur Google Drive dans : agario_rl_backup_v7/eval_match_v7.mp4')

video_path = 'recordings/eval_match_v7.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="850" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(video_path) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Vidéo non trouvée.')

## 7. Exportation Universelle vers ONNX & Benchmark de Latence
Convertit le réseau de neurones PyTorch au standard ONNX ultra-rapide (< 0.02 ms de latence CPU).

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v7/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v6/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip')

valid_cands = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._') and 'bc_pretrained' not in c]
valid_cands.sort(key=extract_step, reverse=True)
best_model = valid_cands[0] if valid_cands else None

if best_model and os.path.exists(best_model):
    print(f'Modèle sélectionné pour l\'export : {best_model}')
    os.makedirs('models', exist_ok=True)
    !python src/inference/export_onnx.py --model "{best_model}" --output models/model_v7.onnx
    if os.path.exists('/content/drive/MyDrive/agario_rl_backup_v7'):
        !cp models/model_v7.onnx /content/drive/MyDrive/agario_rl_backup_v7/model_v7.onnx
        print('📁 Modèle ONNX sauvegardé sur Drive : agario_rl_backup_v7/model_v7.onnx')
else:
    print('⚠️ Aucun checkpoint trouvé pour l\'export ONNX.')